In [1]:
#! .venv/bin/python3
 
import os
# os.chdir('./PhD/TPPRDB_Analysis')
# BERTopic uses tokenisation so throws warnings if multiprocessing after
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import numpy as np
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction import text 
from sklearn.feature_extraction.text import CountVectorizer
import sys
sys.path.append('/Users/jbuc045/Projects/PhD/TPPRDB_Analysis/')
import util_functions as uf
from nltk.corpus import stopwords
import plotly.io as pio
import plotly.graph_objects as go
pio.renderers.default = "browser"
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
import json
from sklearn.metrics import silhouette_score


In [2]:
os.getcwd()
# os.chdir('./PhD/TPPRDB_Analysis')

'/Users/jbuc045/Projects/PhD/TPPRDB_Analysis'

In [2]:
# Prepare data
combined = pd.read_csv("data/mergedDataDec.csv", encoding='utf-8').map(str).map(str.strip).reset_index(drop=True)

# combine title, keywords, abstract, relevance and trace type columns if they are not nan or empty into a single column
combined.index
cols = ['Title', 'Trace_Type','Study_Type', 'Keywords', 'Abstract', 'Exp_Conditions_and_Results',
    'Relevance_to_Canada', 'Addressed question',
    'Activity context', 'Category', 'Specifications',
    'Variables of interest', 'stringency of control', 'No of individuals',
    'Replicates per Individual and condition', 'Nucleic Acid',
    'Bodily origin', 'depositor characteristics',
    'Criteria for shedder status', 'Previous activities',
    'Contact scenario', 'Primary substrate type',
    'Primary substrate Material', 'Deposit', 'Delay (conditions)',
    'Secondary substrate type', 'Secondary Substrate material',
    'Type of secondary contact', 'Further transfer',
    'Background DNA on sampled surface', 'Sampling time',
    'Persistance (conditions)', 'Sampling method', 'Sampling area',
    'Extraction', 'DNA Quantification', 'Input for Profiling', 'Profiling',
    'Reference samples', 'Profile interpretation and mixture analysis',
    'RNA data interpretation', 'DNA Quantitiy', 'Profile Quality',
    'Parameter used for comparison', 'Summary of results',
    'Raised questions (by authors)', 'Cautionary remarks', 'author_keywords']

combined['allData'] = combined[cols].apply(uf._join_non_na, axis=1)

# The sentences to encode
dataAsList = combined['allData'].to_list()

combined.to_csv("modelReadyData.csv")


In [3]:
# Set domain specific stop words (ie forensic, bayesian etc) as too general. At the trace type level we dont care about methods,
# within trace type we can look at these
forensic_stopwords = [
    'forensic', 'bayesian', 'analysis','samples', 'analyses', 'sampled', 'bayes', 'thereom',
    'forensics', 'evidence', 'examination', 'investigation', 'investigations',
    'investigator','sample', 'examined', 'method', 'methods', 'methodology',
    'investigated', 'investigate', 'laboratory', 'laboratories', 'apprehending',
    'research', 'researches', 'case', 'cases', 'casework', 'caseworks', 'testing',
    'examining', 'evaluated', 'evaluation', 'evaluates', 'assessed', 'assessment',
    'crime', 'probabilistic', 'probability', 'probabilities', 'policing'
    'interpretation', 'likelihood', 'ratio','collected','analyze', 'experiments', 
    'analyse', 'experiment', 'analyzed','specimens', 'examiners','analyzing', 'findings', 
    'propositions', 'study', 'techniques', 'technique', 'instrumentation', 'instruments',
    'measurements', 'measurement', 'validation', 'validated', 'validating', 'swabs',
    'swab', 'replicates', 'forensically', 'data', 'results', 'result', 'using', 'used', 
    'use', 'based', 'different', 'assess', 'test', 'tests', 'metholodogies',
    'analysed', 'tested', 'detection', 'detected', 'detect', 'compare',
    'compared', 'comparison', 'comparisons', 'identified', 'identification', 
    'identify','identifies', 'reviewed', 'review', 'reviews', 'obtained', 'obtain',
    'assessing', 'investigations', 'conclusions', 'conclusion', 'concluded',
    'deposited', 'swabbing', 'swabbed', 'studies', 'investigative', 'examines',
    'police', 'officer', 'officers', 'detecting', 'evaluate', 'determine',
    'determining', 'collecting', 'collection', 'analyzes', 'methodologies', 'examine',
    'screening','analysing', 'examinations','evaluating', 'evaluations', 'observations',
    'comparative', 'comparatively', 'detects', 'determined', 'determines', 'investigators',
    'investigates','measure', 'measured', 'measures', 'studied', 'analytical', 'differences'
    'validations', 'validates', 'utilized', 'utilize', 'utilizes', 'documented'
    'characteristics', 'recommendations','factors', 'consideration', 'considerations',
    'investigaating', 'probative','investigating', 'characteristic', 'lab', 'utilizing', 
    'usefulness', 'characterisation', 'characterize', 'characterized', 'characterization', 
    'fbi', 'law enforcement', 'crimes', 'law', 'enforcement', 'practices', 'practice', 
    'caratristiques', 'practise', 'practises', 'security', 'considered', 'consider', 
    'conducted', 'conduct', 'conducts', 'conduction', 'derives', 'derived', 'deriving', 'employed', 
    'employs', 'employing', 'sciences', 'science', 'implementation', 'implementing', 'implemented', 
    'implement', 'involving', 'involved', 'involves', 'utilization', 'utilisations', 'labwork', 
    'laboratorywork', 'laboratoryworks', 'specialized', 'specialise', 'specialises', 'specialised', 
    'practitioner', 'practitioners', 'practising', 'practised', 'authorities', 'authority', 
    'authoritarian', 'regarding', 'regard', 'regards', 'enfsi', 'interpol', 'obtaining', 
    'obtains', 'hypothesis', 'hypotheses', 'evidential', 'evidentially', 'evidences', 
    'operational', 'operations', 'operation', 'operates', 'operate', 'operating', 'standards', 
    'standard', 'standardization', 'standardisations', 'standardise', 'standardises', 
    'standardized', 'protocols', 'protocol', 'procedures', 'procedure', 'procedural', 
    'procedurally', 'practiced', 'criminal', 'criminial', 'criminology', 'criminological',
    'justice', 'search', 'seizure', 'admissibility', 'admissible', 'technology', 'exploitation',
    'exploiting', 'exploited', 'exploits', 'explored', 'normal', 'prepared', 'prepares', 
    'preparing', 'preparation', 'preparations', 'samples analyzed', 'interpretation forensic', 
    'forensic context', 'evaluation forensic', 'feature classification', 'inferences', 'inference',
    'classification', 'interpretation model', 'model interpretation'
    ]

extra_forensic_stopwords = [
    'scene', 'scenes', 'extracted', 'spectrometry', 'extraction', 'chromatography', 
    'analyzer', 'profiling', 'assay', 'assays', 'sampling', 'well', 'analyser', 
    'profile', 'profiles', 'quantification', 'quantified', 'quantify', 'markers',
    'marker', 'spectrometer', 'sequencing', 'sequenced', 'sequencer', 'amplification', 
    'amplified', 'packaging', 'amplify', 'loci', 'locus', 'electrophoresis', 
    'electrophoretic'
                            ]
                      
# Get the list of other language stop words
multilingual_stop_words = stopwords.words()

custom_stopwords = list(text.ENGLISH_STOP_WORDS.union(
    forensic_stopwords,
    extra_forensic_stopwords,
    multilingual_stop_words)
                        )

preprocessed_docs = [uf.preprocess(doc, stopwords=custom_stopwords) for doc in dataAsList]


In [4]:
# create vectorise method
vectorizer_model = CountVectorizer(
    stop_words=custom_stopwords,
    ngram_range=(1, 5),
    min_df=0.2,
    max_df=0.6
)


# Load a pretrained Sentence Transformer model
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2", 
    prompts={
        "classification": "Classify the following text into topics relating to forensic sample types: ",
        "retrieval": "Retrieve semantically similar text, input is in multiple languages: ",
        "clustering": "Identify the topic or theme of forensic sample type or discipline based on the provided \
            text. Do not include law enforcement, justice, statistics or crime as themes. Label topic with trace type or originating location"
    })


In [5]:
# Calculate embeddings by calling model.encode() saves time later for BERTopic to avoid doing this internally
embeddings = model.encode(preprocessed_docs)
print(embeddings.shape)
# should be same rows as data ie [3279, 384]


(3279, 768)


Topic Modeling with BERTopic: Minimum Viable Example

References:

1 https://maartengr.github.io/BERTopic/getting_started/embeddings/embeddings.html
2 https://maartengr.github.io/BERTopic/getting_started/clustering/clustering.html
3 https://maartengr.github.io/BERTopic/getting_started/visualization/visualization.html

In [6]:
# Fine-tune the topic representations
representation_model = KeyBERTInspired(
    top_n_words=100,
    nr_repr_docs=800,
    nr_samples=2000,
    nr_candidate_words=2000)


In [7]:
# macOS has a bug with matrix multiplication that causes runtime warnings of zero division. 
    # Create custom HDBSCAN model
hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    min_samples=15,
    cluster_selection_epsilon=0.1,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [8]:
# Bertopic model instantiation
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    embedding_model = model,
    calculate_probabilities=True,
    nr_topics='auto',
    #seed_topic_list=topic_list
    )

topics,probs = topic_model.fit_transform(dataAsList, embeddings=embeddings)
new_topics = topic_model.reduce_outliers(dataAsList,
                                        topic_model.topics_, # type: ignore
                                        strategy="probabilities",
                                        probabilities=topic_model.probabilities_ ) # type: ignore

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



In [11]:
# load optimised model
# best_model=BERTopic.load("bertopic_optimized_hdbscan", embedding_model=model)

with open("best_model_params.json", "r") as f:
    best_results = json.load(f)

combined['T1_topic'] = pd.Series(new_topics) #best_results[1]

#combined['T1_probs'] = best_results[2] # type: ignore

In [31]:
#topic_model= best_model

topic_info = topic_model.get_topic_info() # type: ignore
#topic_model.set_topic_labels(list(topic_info['Name']))

# Exclude the -1 topic (outliers) for labeling the main topics
topic_info = topic_info[topic_info['Topic'] != -1].reset_index(drop=True)

# associate colours with the topics
colours_list = [
    '#d62728','#f3722c','#ffa15a','#fecb52','#f5f511',
    '#b6e880','#2ca02c','#4d908e','#43aa8b','#00cc96','#4cddc9',
    '#90dbf4','#19d3f3','#005073','#718591','#1f77b4','#a3c4f3','#cfbaf0',
    '#6a19b5','#636efa','#ab63fa','#b9fbc0','#ff97ff','#ffcfd2'
    ]

colour_dict=dict(zip(topic_info['Representation'].astype(str).tolist(),colours_list))

mapped_colours = topic_info['Representation'].astype(str).map(colour_dict).tolist() 

In [32]:
# intertopic distance map
dist_map = topic_model.visualize_topics(width =1250) # type: ignore

# make coords for the annotations. Add offset to x placement to avoid overlap
coords = np.column_stack((dist_map.data[0]['x'], dist_map.data[0]['y'])) # type: ignore

In [37]:
# add random offset to extracted coords
sizeref = dist_map.data[0]['marker']['sizeref'] # type: ignore
offset_x = coords[:,0] + (sizeref * np.random.normal(0,1.5,len(coords[:,0]))) # type: ignore
    # ( * sizeref * 2) + # type: ignore
    # (np.random.binomial(1,0.5) * sizeref * -2) ) # type: ignore
offset_y=coords[:,1] + (sizeref * np.random.normal(0,1.5,len(coords[:,1]))) # type: ignore
    # (np.random.binomial(1,0.5) * sizeref* 6) + # type: ignore
    # (np.random.binomial(1,0.5) * sizeref * -4) ) # type: ignore

positions = np.column_stack([offset_x, offset_y])

min_dist = 1.8 # minimum allowed distance between any label and any other object

positions = uf.resolve_overlaps(positions, coords, min_dist, repulsion_strength=0.5,
        marker_repulsion_strength=1, attraction_strength=0.08, max_step=2)

offset_x, offset_y = positions[:, 0], positions[:, 1]

# Add static labels as annotations to the Plotly figure
annotations = []
for index, row in topic_info.iterrows():

    annotations.append(
        dict(
            ax=offset_x[index], # arrow tail pos # type: ignore
            ay=offset_y[index], # type: ignore
            text=f"Topic {index}", # Use topic name
            showarrow=True,
            x=coords[index][0], #arrow head pos
            y=coords[index][1],
            font=dict(size=10, color="black"),
            # Position the text slightly offset from the marker
            xanchor='left',
            yanchor='middle',
            xref="x",
            yref="y",
            arrowhead=1,
            axref= 'x',
            ayref='y'
            )
    )
#remove slider first so it only prints to console once
dist_map['layout'].pop('sliders')

#update markers with colours
dist_map.update_traces(
    marker=dict(color=mapped_colours ),
    selector=dict(mode='markers'),
    # name=topic_info['Name'].tolist()
)

# update to add the static labels
dist_map.update_layout(
    annotations=annotations,               
    showlegend=True,
    legend=dict(
        orientation="v",
        yanchor="bottom",
        x=1.02,
        xanchor="right",
        y=1
    ),
    title={
        'text': "Distance Map of Topics",
        'y':0.99,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    }
    )

# Display the figure
dist_map.show()

In [38]:
from IPython.display import display, HTML
displayTable = topic_info[['Topic','Name','Count']]
displayTable['Name'] = displayTable['Name'].str.replace("_", " | ")
displayTable['Name'] = displayTable['Name'].str.replace(r"\d+ \| ", "", regex=True)
styledTable = displayTable.style.set_table_styles([
    # Apply to Headers (th): Borders and Center Alignment
    {'selector': 'th', 'props': [
        ('background-color', 'white'),
        ('border-style', 'solid'),
        ('border-width', '1px'),
        ('border-color', 'black'),
        ('text-align', 'center'),
        ('font-weight', 'bold')
    ]},
    # Apply to Data Cells (td): Borders and Center Alignment
    {'selector': 'td', 'props': [
        ('background-color', 'white'),
        ('border', '1px solid black'),
        ('border-right', '1px solid black'),
        ('border-top', 'none'),
        ('border-bottom', 'none'),
        ('text-align', 'center') 
    ]}
]).hide()

# display table
styledTable

Topic,Name,Count
0,transfer skin | contamination transfer | trace skin | trace skin hands,502
1,sexual assault vaginal | sexual assault skin | assault vaginal | persistence sexual assault,182
2,fluids blood saliva | bodily fluids bloodstain persistence | stains bodily fluids | blood applied,179
3,mixture approach | approach mixtures | contributor mixture | contributor mixed,51
4,hairs | hair preliminary | hair hair | hair shafts,41
5,genotypes | human genomic | genetics | population genetics,37
6,dental | teeth | oral | bite mark,35
7,bloodstain bloodstain pattern | fluids bloodstain bloodstain pattern | pattern bloodstain pattern | patterns bloodstain,34
8,condoms | condom | persistence vaginal | sexual assault vaginal,31
9,transfer persistence prevalence recovery | transfer persistence prevalence | background recovery persistance primary | success rates,26


In [39]:
#save table to file
import imgkit
imgkit.from_string(styledTable.to_html(), 'BioTopicTable15ngramAllStops.png')

Loading page (1/2)
Rendering (2/2)                                                    
Done                                                               


True

In [22]:
# topic-terms barcharts as HTML file
bar_fig = topic_model.visualize_barchart(top_n_topics=26, autoscale=True, width=350) # type: ignore
bar_fig.show()

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(preprocessed_docs) # type: ignore

# Save topics dendrogram as HTML file
hierarch_fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics) # type: ignore
hierarch_fig.show()#.write_html("./PhD-Windows/TPPRDB_Analysis/hieararchy.html")

In [ ]:
# # Reduce dimensionality of embeddings, this step is optional but much faster to perform iteratively:
# reduced_embeddings = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='cosine').fit_transform(embeddings)
# reduced_topics = topic_model.visualize_documents(dataAsList, reduced_embeddings=np.array(reduced_embeddings))
# #.write_html("./PhD-Windows/TPPRDB_Analysis/reduced_projections.html")
# reduced_topics.show()

# time series analysis

# The data to encode
# datedData = combined[['Year','Title','Trace_Type','Keywords','Abstract','Exp_Conditions_and_Results','Relevance_to_Canada']]
# dataAsDatedList = datedData.apply(
#     lambda row: '; '.join(row.dropna().astype(str)), axis=1
# ).to_list()

# date = datedData.Year
# date[date=='s.d.'] = np.nan


# topics_over_time = topic_model.topics_over_time(dataAsDatedList, date.astype('str').to_list())
# model.visualize_topics_over_time(topics_over_time, topics=[range(1,21)])

# Save the model
# topic_model.save("models/TPPRDB_BERTopic_Model")    
# Save model
topic_model.save("bertopic_optimized_hdbscan_15ngramAllStops",# type: ignore
                serialization="safetensors",
                save_embedding_model=True,
                save_ctfidf=True) 
import json

# A function to handle NumPy types during JSON serialization
def numpy_encoder(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    # If the object is not a type we handle, raise a TypeError
    # The default encoder will then handle standard types
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")



with open("best_model15ngramAllStops_params.json", "w") as f:
    json.dump(best_results, f, default=numpy_encoder )

In [26]:
# subset data to that assigned to topic 1 (trace DNA) and rerun process
BioData = combined[combined['T1_topic'].isin([1,2,7,8,11,13,14,15,17,20,24,25])]

In [ ]:
# make a list
bioDataAsList = BioData['allData'].to_list()

# add extra DNA specific stop words
dna_stopwords = [
    'dna', 'deoxyribonucleic', 'nucleic', 'acid', 'profile', 'profiles', 'genetic',
    'genome',  'genomes', 'genotyping', 'genotype', 'sequencing', 'sequence', 'abi'
    'sequences', 'str', 'loci', 'locus', 'microsatellite', 'microsatellites',
    'amplification', 'pcr', 'amplified', 'amplify', 'electrophoresis', 'electrophoretic',
    'quantification', 'quantified', 'quantify', 'markers', 'marker', 'alleles', 'allele',
    'profiling', 'extraction', 'extracted', 'extract', 'swab', 'swabs', 'replicate',
    'replicates', 'replication', 'ngs', 'mitochondrial', 'mitochondria', 'y-chromosome',
    'autosomal', 'autosome', 'locus', 'loci', 'capillary', 'capillaries', 'interpretation'
]   

all_stopwords = list(text.ENGLISH_STOP_WORDS.union(
    forensic_stopwords,
    dna_stopwords,
    extra_forensic_stopwords,
    multilingual_stop_words)
                        )
# preprocess
bioPreprocessed_docs = [uf.preprocess(doc, stopwords=all_stopwords) for doc in bioDataAsList]

In [53]:
bioEmbeddings = model.encode(bioDataAsList)

In [54]:
Biotopics,Bioprobs = topic_model.fit_transform(bioDataAsList, embeddings=bioEmbeddings)
# new_topics = topic_model.reduce_outliers(preprocessed_docs,
#                                         topic_model.topics_, # type: ignore
#                                         strategy="probabilities",
#                                         probabilities=topic_model.probabilities_ ) # type: ignore

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



In [30]:
BioData['T2_topic'] = Biotopics #best_results[1]

In [ ]:
param_grid = [
    {"min_cluster_size": i, "min_samples": j, "cluster_selection_epsilon": k} 
    for i in range(5,26,5) 
    for j in [1,5, 10, 15]
    for k in uf.float_range(0,0.3,0.1)]

results = []
best_model = None
best_results = []
best_score = -np.inf



In [16]:

# Evaluate each configuration
for params in param_grid:
    print(f"\nTraining with params: {params}")

    # Create custom HDBSCAN model
    hdbscan_model = HDBSCAN(
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
        cluster_selection_epsilon=params["cluster_selection_epsilon"],
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True
    )

    # Bertopic model instantiation
    topic_model = BERTopic(
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        embedding_model = model,
        calculate_probabilities=True,
        nr_topics='auto',
        #seed_topic_list=topic_list
        )

    topics,probs = topic_model.fit_transform(preprocessed_docs, embeddings=embeddings)
    new_topics = topic_model.reduce_outliers(preprocessed_docs,
                                        topic_model.topics_, # type: ignore
                                        strategy="probabilities",
                                        probabilities=topic_model.probabilities_ ) # type: ignore

    # Evaluate model
    # Topic coherence
    coherence = uf.calculate_coherence_score(topic_model, preprocessed_docs)

    # Topic diversity
    topic_words = topic_model.get_topics()
    diversity = uf.calculate_diversity_score(topic_model)

    # Silhouette score (only on clustered docs)
    valid_idx = [i for i, t in enumerate(topics) if t != -1]
    if len(valid_idx) > 2:
        sil_score = silhouette_score(
            np.array(embeddings)[valid_idx],
            np.array(topics)[valid_idx]
        )
    else:
        sil_score = -1

    score = coherence * 0.7 + diversity * 0.15 + sil_score * 0.15  # weighted scoring

    print(f"Coherence={coherence:.4f}, Diversity={diversity:.4f}, Silhouette={sil_score:.4f}, Score={score:.4f}")

    results.append((params, coherence, diversity, sil_score, score))

    # Keep best model
    if score > best_score:
        best_results = [params, new_topics, probs, coherence, diversity, sil_score, score]
        best_score= score
        best_model = topic_model
        

# Report & use best model
print("\nBest configuration:", best_model.hdbscan_model.get_params()) # type: ignore
print("Best score:", best_score)


Training with params: {'min_cluster_size': 5, 'min_samples': 1, 'cluster_selection_epsilon': 0}


/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



Coherence=-0.4533, Diversity=0.6004, Silhouette=0.0635, Score=-0.2177

Training with params: {'min_cluster_size': 5, 'min_samples': 1, 'cluster_selection_epsilon': 0.1}


/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



Coherence=-0.4480, Diversity=0.6656, Silhouette=0.0371, Score=-0.2082

Training with params: {'min_cluster_size': 5, 'min_samples': 1, 'cluster_selection_epsilon': 0.2}


...

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



Coherence=-0.4204, Diversity=0.8227, Silhouette=0.0613, Score=-0.1617

Training with params: {'min_cluster_size': 25, 'min_samples': 15, 'cluster_selection_epsilon': 0.2}


/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



Coherence=-0.4360, Diversity=0.7018, Silhouette=0.0348, Score=-0.1947

Best configuration: {'algorithm': 'best', 'allow_single_cluster': False, 'alpha': 1.0, 'approx_min_span_tree': True, 'branch_detection_data': False, 'cluster_selection_epsilon': 0.0, 'cluster_selection_epsilon_max': inf, 'cluster_selection_method': 'eom', 'core_dist_n_jobs': 4, 'gen_min_span_tree': False, 'leaf_size': 40, 'match_reference_implementation': False, 'max_cluster_size': 0, 'memory': Memory(location=None), 'metric': 'euclidean', 'min_cluster_size': 10, 'min_samples': None, 'p': None, 'prediction_data': True}
Best score: -0.15292317252251117


/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



In [ ]:
results_df = pd.DataFrame(results, columns=["Parameters", "Coherence", "Diversity", "Silhouette", "Score"])
results_df#.sort_values(by='Score', ascending=False)

In [ ]:
results_df.plot(kind='bar', figsize=(40,6), title='Model Evaluation Scores for Different HDBSCAN Configurations',)

In [ ]:
[{'min_cluster_size': 25, 'min_samples': 15, 'cluster_selection_epsilon': 0.1},
 [5,  6,  np.int64(1),  10,  np.int64(1),  3,  2,  np.int64(1),  np.int64(1),  15,  1,  np.int64(1),  np.int64(1),  1,
  6,  9,  14,  0,  5,  10,  14,  3,  6,  np.int64(1),  1,  np.int64(1),  0,  1,  2,  1,  1,  2,  np.int64(0),  4,  8,  
  9,  1,  5,  2,  1,  0,  np.int64(1),  5,  1,  1,  3,  3,  3,  2,  np.int64(1),  2,  5,  np.int64(0),  0,  3,  np.int64(0),
  12,  3,  5,  np.int64(0),  12,  0,  2,  1,  3,  np.int64(6),  np.int64(1),  2,  0,  1,  1,  2,  2,  4,  2,  10,  
  np.int64(0),  np.int64(1),  0,  0,  0,  2,  3,  np.int64(1),  0,  6,  6,  2,  1,  1,  7,  10,  np.int64(1),  2,  1,  8,  2,
  0,  2,  0,  0,  3,  np.int64(0),  0,  2,  0,  np.int64(0),  np.int64(0),  14,  1,  np.int64(1),  np.int64(1),  12,  12,
  np.int64(1),  np.int64(2),  np.int64(0),  0,  np.int64(0),  np.int64(2),  0,  1,  np.int64(0),  11,  np.int64(1),  3,  2, 
  3,  3,  3,  6,  np.int64(0),  3,  14,  14,  0,  14,  1,  12,  6,  np.int64(1),  np.int64(1),  0,  0,  2,  14, 2,  0,  0,
  0,  6,  4,  2,  np.int64(0),  1,  np.int64(2),  2,  np.int64(7),  np.int64(0),  0,  0,  10,  2,  2,  2,  0,  0,  np.int64(0),
  0,  2,  1,  0,  np.int64(1),  0,  np.int64(1),  0,  np.int64(1),  0,  np.int64(0),  np.int64(0),  0,  0, 1,  7,  7,  0,  0,  
  np.int64(2),  np.int64(1),  np.int64(7),  3,  1,  np.int64(0),  np.int64(0),  3,  3,  1,  np.int64(0),  4,  4,  5,  4,  6,  6,
  3,  6,  0,  np.int64(0),  np.int64(0),  0,  3,  1,  6,  8,  np.int64(1),  6,  3,  7,  0,  3,  2,  1,  6,  np.int64(7),  4,  0,  
  4,  1,  0,  6,  6,  6,  0,  6,  4,  np.int64(0),  6,  5,  6,  4,  4,  np.int64(6),  4,  1,  3,  3,  1,  1,  1,  3,  3,  8,
  2,  6,  3,  3,  12,  1,  0,  3,  2,  np.int64(1),  np.int64(1),  0,  2,  0,  0,  0,  np.int64(1),  1,  10,  np.int64(1),  1,  
  np.int64(0),  6,  10,  6,  10,  2,  0,  np.int64(7),  np.int64(0),  0,  2,  16,  np.int64(1),  13,  1,  5,  np.int64(7),  np.int64(0),
  np.int64(1),  8,  10,  np.int64(2),  2,  0,  np.int64(1),  0,  0,  2,  2,  7,  2,  0,  1,  1,  0,  0,  np.int64(0),  5,  np.int64(1),
  4,  2,  3,  3,  7,  13,  12,  np.int64(1),  0,  12,  2,  0,  1,  5,  1,  0,  0,  1,  3,  1,  1,  np.int64(7),  0,  0,  0,  0,  5,
  np.int64(0),  0,  2,  1,  1,  0,  1,  1,  np.int64(0),  6,  2,  1,  3,  np.int64(1),  3,  0,  np.int64(1),  0,  np.int64(1),  1,  2, 
  0,  9,  2,  1,  0,  0,  9,  5,  3,  np.int64(0),  np.int64(1),  1,  12,  4,  4,  np.int64(1),  2,  4,  10,  np.int64(1),  np.int64(7), 
  2,  2,  4,  np.int64(0),  4,  2,  0,  1,  14,  7,  4,  0,  2,  1,  1,  0,  0,  np.int64(1),  7,  np.int64(1),  0,  1,  0,  0,  
  np.int64(6),  np.int64(6),  0,  np.int64(0),  np.int64(0),  np.int64(0),  np.int64(0),  np.int64(0),  0,  7,  np.int64(1),  0,  0,  
  8,  8,  0,  0,  0,  16,  2,  2,  np.int64(1),  np.int64(1),  np.int64(0),  np.int64(0),  0,  np.int64(0),  5,  np.int64(1),  np.int64(0), 
  0,  1,  1,  5,  np.int64(1),  0,  np.int64(1),  0,  4,  np.int64(18),  np.int64(1),  0,  np.int64(1),  0,  7,  13,  0,  1,  0,  12,  4,  
  1,  2,  4,  0,  2,  np.int64(1),  1,  3,  np.int64(7),  0,  0,  np.int64(1),  2,  4,  0,  4,  8,  2,  np.int64(0),  0,  0,  0,  11,  
  np.int64(1),  0,  1,  5,  9,  11,  17,  13,  0,  1,  1,  np.int64(1),  np.int64(1),  1,  1,  9,  1,  0,  2,  1,  5,  5,  1,  1,  6,  6,
  1,  0,  np.int64(1),  0,  1,  1,  1,  2,  np.int64(0),  5,  np.int64(1),  np.int64(1),  1,  np.int64(0),  17,  2,  11,  0,  0,  1,  3,
  np.int64(1),  0,  9,  np.int64(1),  np.int64(1),  0,  0,  5,  np.int64(0),  np.int64(0),  4,  np.int64(1),  0,  2,  3,  4,  3,  0,  2,  
  2,  0,  6,  15,  0,  2,  np.int64(1),  3,  4,  2,  3,  3,  6,  6,  1,  1,  1,  7,  0,  2,  2,  16,  2,  np.int64(1),  3,  3,  0,  3,  
  np.int64(1),  np.int64(1),  np.int64(1),  1,  np.int64(1),  11,  1,  1,  10,  np.int64(1),  np.int64(0),  6,  6,  6,  6,  0,  12,  0,
  np.int64(0),  0,  np.int64(0),  9,  9,  4,  2,  2,  0,  0,  6,  1,  np.int64(1),  np.int64(0),  1,  0,  8,  15,  5,  5,  7,  np.int64(7),
  np.int64(7),  4,  9,  0,  2,  17,  4,  0,  17,  0,  13,  5,  0,  1,  np.int64(0),  11,  2,  np.int64(1),  np.int64(1),  0,  7,  5,  np.int64(1),
  np.int64(6),  0,  4,  np.int64(1),  1,  4,  np.int64(0),  7,  np.int64(0),  4,  0,  10,  4,  0,  14,  4,  1,  0,  0,  np.int64(0),  0,  0,
  np.int64(1),  0,  0,  4,  0,  0,  0,  np.int64(0),  14,  2,  np.int64(0),  0,  0,  np.int64(0),  np.int64(0),  0,  0,  np.int64(1),  4,
  14,  np.int64(1),  1,  np.int64(1),  0,  0,  15,  1,  0,  4,  8,  11,  np.int64(0),  2,  0,  1,  np.int64(1),  9,  2,  np.int64(1),  np.int64(0),
  3,  3,  9,  7,  7,  np.int64(7),  7,  7,  np.int64(7),  7,  np.int64(7),  7,  np.int64(7),  7,  9,  6,  1,  1,  1,  10,  1,  np.int64(0),  1,
  3,  10,  2,  np.int64(1),  np.int64(1),  4,  np.int64(1),  0,  np.int64(1),  2,  np.int64(0),  0,  18,  15,  np.int64(1),  4,  1,  
  np.int64(0),  8,  8,  9,  0,  np.int64(1),  0,  np.int64(1),  np.int64(0),  0,  np.int64(7),  np.int64(1),  0,  0,  2,  0,  np.int64(0),
  np.int64(7),  np.int64(1),  1,  7,  4,  1,  np.int64(0),  0,  14,  1,  1,  1,  1,  np.int64(1),  0,  13,  13,  9,  11,  np.int64(6),
  np.int64(0),  np.int64(0),  15,  4,  np.int64(1),  0,  5,  0,  14,  np.int64(1),  8,  8,  8,  2,  10,  np.int64(1),  2,  1,  1,  1,  7,  1,
  0,  0,  0,  0,  np.int64(1),  0,  4,  2,  2,  0,  1,  0,  9,  np.int64(0),  12,  12,  np.int64(1),  1,  4,  3,  3,  0,  0,  0,  0,  2,  2,
  np.int64(7),  4,  np.int64(10),  np.int64(1),  np.int64(1),  17,  np.int64(1),  11,  2,  2,  2,  np.int64(1),  1,  np.int64(6),  1,  np.int64(1),
  0,  np.int64(0),  1,  2,  1,  1,  np.int64(7),  0,  0,  15,  0,  0,  1,  np.int64(1),  8,  7,  7,  0,  0,  1,  7,  np.int64(7),  1,  2,  0,
  0,  0,  0,  0,  0,  np.int64(0),  np.int64(0),  0,  15,  0,  0,  0,  0,  np.int64(0),  np.int64(0),  0,  np.int64(0),  np.int64(0),  0,
  np.int64(0),  0,  np.int64(0),  0,  0,  1,  1,  np.int64(1),  1,  1,  7,  0,  np.int64(0),  0,  0,  np.int64(1),  np.int64(1),  0,  0,
  0,  0,  2,  0,  2,  np.int64(1),  2,  0,  0,  0,  7,  7,  7,  0,  np.int64(1),  np.int64(6),  0,  0,  5,  0,  7,  0,  0,  np.int64(1),  
  np.int64(7),  0,  np.int64(1),  7,  np.int64(0),  3,  np.int64(0),  0,  9,  np.int64(0),  2,  2,  15,  16,  np.int64(7),  7,  7,  7,  
  np.int64(1),  np.int64(1),  2,  np.int64(0),  np.int64(2),  0,  np.int64(0),  7,  np.int64(1),  0,  0,  2,  np.int64(1),  np.int64(2),  
  np.int64(2),  12,  1,  1,  0,  1,  2,  2,  np.int64(1),  2,  5,  2,  1,  5,  5,  3,  3,  0,  2,  np.int64(0),  np.int64(0),  8,  0,  8,  8,
  8,  10,  np.int64(1),  2,  6,  14,  6,  np.int64(7),  2,  2,  2,  2,  np.int64(0),  1,  2,  1,  ...],
 array([[0.05661714, 0.1477075 , 0.04493778, ..., 0.01833907, 0.01068646,
         0.01098123],
        [0.02357403, 0.03620924, 0.01033425, ..., 0.00288404, 0.0025419 ,
         0.00273146],
        [0.03496197, 0.33002632, 0.10540221, ..., 0.01471195, 0.01136718,
         0.01067791],
        ...,
        [0.07028602, 0.22543384, 0.06178239, ..., 0.01715875, 0.01483812,
         0.01267124],
        [0.01071972, 0.02800483, 0.00894951, ..., 0.00154066, 0.00261808,
         0.00139168],
        [0.00261908, 0.00685734, 0.0021926 , ..., 0.00037669, 0.00064062,
         0.0003399 ]], shape=(3279, 19)),
 np.float64(-0.17087368158910116),
 0.669,
 0.06790818274021149,
 np.float64(-0.009075349701339072)]

In [ ]:
# bio data with stop words
print(uf.calculate_coherence_score(topic_model, bioPreprocessed_docs))
print(uf.calculate_diversity_score(topic_model))
print(silhouette_score(bioEmbeddings, Biotopics))

-0.44411482408474706
0.7807142857142857
0.0011433713370934129


/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



In [ ]:
# data with stopwords
print(uf.calculate_coherence_score(topic_model, preprocessed_docs))
print(uf.calculate_diversity_score(topic_model))
print(silhouette_score(embeddings, topics))

-0.4494435792738913 0.7072222222222222 0.0065061720088124275


/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



In [52]:
#data without stopwords
print(uf.calculate_coherence_score(topic_model, dataAsList))
print(uf.calculate_diversity_score(topic_model))
print(silhouette_score(embeddings, topics))

-0.28113759145381617
0.5436363636363636
0.016673358157277107


/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



In [ ]:
print(uf.calculate_coherence_score(topic_model, bioDataAsList))
print(uf.calculate_diversity_score(topic_model))
print(silhouette_score(bioEmbeddings, Biotopics))

-0.3156524630139563
0.5577777777777778
0.007735783699899912


/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/jbuc045/Projects/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



In [19]:
# Feature importance in clustering
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier


x_features = combined.loc[:,['Title', 'Trace_Type','Study_Type', 'Keywords', 'Abstract', 'Exp_Conditions_and_Results',
    'Relevance_to_Canada', 'Addressed question',
    'Activity context', 'Category', 'Specifications',
    'Variables of interest', 'stringency of control', 'No of individuals',
    'Replicates per Individual and condition', 'Nucleic Acid',
    'Bodily origin', 'depositor characteristics',
    'Criteria for shedder status', 'Previous activities',
    'Contact scenario', 'Primary substrate type',
    'Primary substrate Material', 'Deposit', 'Delay (conditions)',
    'Secondary substrate type', 'Secondary Substrate material',
    'Type of secondary contact', 'Further transfer',
    'Background DNA on sampled surface', 'Sampling time',
    'Persistance (conditions)', 'Sampling method', 'Sampling area',
    'Extraction', 'DNA Quantification', 'Input for Profiling', 'Profiling',
    'Reference samples', 'Profile interpretation and mixture analysis',
    'RNA data interpretation', 'DNA Quantitiy', 'Profile Quality',
    'Parameter used for comparison', 'Summary of results',
    'Raised questions (by authors)', 'Cautionary remarks', 'author_keywords']] 
    
X_train, X_test, y_train, y_test = train_test_split(x_features,combined['T1_topic'], test_size=0.2, random_state=0)

knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_train, y_train)
knn_preds = knn.predict(X_test)
knn_acc = accuracy_score(y_test, knn_preds)
knn_cm = confusion_matrix(y_test, knn_preds)
 


ValueError: could not convert string to float: 'Calculation Of Likelihood Ratios In Forensic Glass Comparisons; Introduction To A R Code And Shiny App Applied To Existing Background Glass Elemental Databases'